# Métricas de calidad, costo y rendimiento

Este notebook enseña que la observabilidad no es solo ver logs: también es medir la calidad, la eficiencia y el costo de un agente LLM.

## Objetivos
- Medir latencia, tokens, costo y errores.
- Evaluar calidad mediante una rúbrica simple.
- Comparar dos versiones de prompt.
- Detectar regresiones entre versiones.

In [ ]:
!pip install pandas langchain langchain-openai wikipedia

In [ ]:
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

In [ ]:
import random
import time
from pprint import pprint

try:
    import pandas as pd
except ImportError:
    raise ImportError('Instala pandas con `pip install pandas` antes de ejecutar este notebook.')

# Dataset de 10 preguntas de ejemplo
questions = [
    '¿Cuál es la capital de Argentina?',
    'Explícame qué es la observabilidad en LLMs.',
    'Calcula 12 * 7.',
    'Busca la mejor receta de guacamole.',
    '¿Qué hizo Nelson Mandela?',
    'Resume las ventajas de usar memoria en agentes.',
    'Define qué es un span en trazabilidad.',
    '¿Cómo funciona una batería de auto eléctrico?',
    'Dime un dato sobre el clima actual.',
    '¿Qué es una regresión de prompt?'
]

In [ ]:
# Definición de cómputo de métricas y evaluaciones
def simulate_response(question, prompt_version):
    # Simulamos latencia y uso de tokens según la versión de prompt
    base_latency = 0.8 if prompt_version == 'prompt_v1' else 1.1
    noise = random.uniform(-0.15, 0.15)
    latency = max(0.2, base_latency + noise)
    tokens_in = len(question.split()) + (10 if prompt_version == 'prompt_v2' else 6)
    tokens_out = random.randint(40, 80) if prompt_version == 'prompt_v2' else random.randint(20, 60)
    cost = round((tokens_in + tokens_out) * 0.00002, 6)

    if prompt_version == 'prompt_v2' and 'regresión' in question.lower():
        accuracy = 1.0
        grounded = 1.0
    else:
        accuracy = random.choice([0.7, 0.8, 0.9]) if prompt_version == 'prompt_v2' else random.choice([0.5, 0.6, 0.7])
        grounded = random.choice([0.7, 0.8, 0.9]) if prompt_version == 'prompt_v2' else random.choice([0.5, 0.6, 0.7])

    error = None if accuracy >= 0.7 else 'respuesta incompleta'
    response = f'Respuesta simulada ({prompt_version}): {question[:50]}...'

    return {
        'question': question,
        'prompt_version': prompt_version,
        'response': response,
        'latency_s': round(latency, 3),
        'tokens_in': tokens_in,
        'tokens_out': tokens_out,
        'cost_estimated': cost,
        'accuracy': accuracy,
        'groundedness': grounded,
        'error': error,
    }

def evaluate_result(row):
    # Rúbrica simple de calidad basada en accuracy/groundedness
    score = round((row['accuracy'] + row['groundedness']) / 2 * 100)
    return {
        'accuracy': row['accuracy'],
        'groundedness': row['groundedness'],
        'quality_score': score,
        'usable': row['accuracy'] >= 0.7 and row['groundedness'] >= 0.7,
    }

In [ ]:
results = []
for prompt_version in ['prompt_v1', 'prompt_v2']:
    for q in questions:
        row = simulate_response(q, prompt_version)
        metrics = evaluate_result(row)
        results.append({**row, **metrics})

df = pd.DataFrame(results)
display(df.head(10))

summary = df.groupby('prompt_version').agg({
    'accuracy': 'mean',
    'latency_s': 'mean',
    'tokens_in': 'mean',
    'tokens_out': 'mean',
    'cost_estimated': 'sum',
    'usable': 'sum',
}).rename(columns={'usable': 'usable_count'})
summary['latencia_promedio'] = summary['latency_s'].round(3)
summary['tokens_promedio'] = ((summary['tokens_in'] + summary['tokens_out']) / 2).round(1)
summary['accuracy'] = (summary['accuracy'] * 100).round(1)
summary['costo_total'] = summary['cost_estimated'].round(6)
summary = summary[['accuracy', 'latencia_promedio', 'tokens_promedio', 'costo_total', 'usable_count']]
display(summary)

## Interpretación de métricas

- `accuracy` y `groundedness` son métricas de calidad.
- `latencia_s`, `tokens_in` y `tokens_out` son métricas de rendimiento.
- `cost_estimated` ayuda a comparar versiones de prompt desde el punto de vista económico.
- `usable_count` muestra cuántas respuestas cumplen la rúbrica mínima.

Esta comparación ayuda a tomar decisiones cuando un prompt mejorado aumenta la calidad pero también eleva el costo.

## Ejemplo de comparación final

| versión | accuracy (%) | latencia promedio | tokens promedio | costo total | respuestas útiles |
|---|---|---|---|---|---|
| prompt_v1 | valor | valor | valor | valor | valor |
| prompt_v2 | valor | valor | valor | valor | valor |

Esta tabla ejemplifica cómo una métrica compuesta es más útil que la sola apariencia de una respuesta.

: {
: {
: 
3
, 
: 
, 
: 
},
: {
: 
}
: 4,
: 5